# Ten repeated runs with a GROUPED partition (near-duplicate images always on the same side)

**Why this notebook exists.** In the earlier partition, each image was drawn on its own for training, validation, or test. Because the same photograph appears in files with different identifiers, about 38% of the test images had a near-duplicate in the training or validation subsets. Here, each **group of near-duplicate images** is allocated as a whole to a single subset.

**What does not change:** pre-trained YOLOv8n-cls, 30 epochs, patience 10, `imgsz=224`, the same 10 seeds, and the 70/15/15 split per cultivar (1,031 / 220 / 225 images).

**What changes:** only which images go to which subset. Results are written to a new folder (`runs_agrupado`) and do not overwrite earlier ones.

**How to use**
1. Run the cells of **Part A** (fast, no GPU). They build the partitions and report the leakage under the earlier image-level partition and under the grouped partition. Check this output before training.
2. Run **Part B** (training; GPU required; several hours). If Colab disconnects, run it again: runs that are already complete are skipped.
3. **Part C** (color baselines with SVM and Random Forest) uses the same partitions and needs no GPU.

**Language note.** Comments, messages, and text are in English. File names and column names of the output tables (for example `classe`, `arquivo`, `grupo`, `particao`, `resumo_por_rodada.csv`) are kept as originally generated, so that they match the files deposited with the paper.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ultralytics version used for the reported results (falls back to the latest if it is no longer available)
!pip install -q ultralytics==8.4.208 || pip install -q ultralytics

In [ ]:
import os, glob, json, random, shutil, math
import numpy as np, pandas as pd, scipy.stats as st

# ======================= CONFIGURATION =======================
DATASET_ORIGINAL = '/content/drive/MyDrive/Colab_Notebooks/four_varieties/data_cropped'    # same folder as the earlier 10 runs
BASE_WORK_DIR    = '/content/drive/MyDrive/Colab_Notebooks/four_varieties/runs_agrupado'   # NEW folder
GRUPOS_CSV       = None        # None = search Drive automatically for 'dimensoes_grupos.csv'
SEEDS  = [42, 101, 2023, 777, 88, 1234, 555, 999, 333, 111]
EPOCHS, PATIENCE = 30, 10
SPLIT_RATIO = (0.7, 0.15, 0.15)
IMG_EXT = ('.jpg', '.jpeg', '.png')
REF_SEED = 101                 # reference run of the paper (chosen beforehand, without looking at the test results)
# ============================================================

assert os.path.isdir(DATASET_ORIGINAL), f'Folder not found: {DATASET_ORIGINAL}. Adjust DATASET_ORIGINAL.'
os.makedirs(BASE_WORK_DIR, exist_ok=True)
if GRUPOS_CSV is None:
    achados = sorted(glob.glob('/content/drive/MyDrive/**/dimensoes_grupos.csv', recursive=True))
    assert achados, 'dimensoes_grupos.csv not found. Run looking_for_duplicates.ipynb first, which creates the near-duplicate groups.'
    if len(achados) > 1: print('Warning: several files found:', achados)
    GRUPOS_CSV = achados[0]
print('Groups read from:', GRUPOS_CSV)

G = pd.read_csv(GRUPOS_CSV)
G['arquivo'] = G['file'].map(os.path.basename)
CLASSES = sorted(d for d in os.listdir(DATASET_ORIGINAL) if os.path.isdir(os.path.join(DATASET_ORIGINAL, d)))
ALL = {c: sorted(f for f in os.listdir(os.path.join(DATASET_ORIGINAL, c)) if f.lower().endswith(IMG_EXT)) for c in CLASSES}
GRUPO = {(r.classe, r.arquivo): r.grupo for r in G.itertuples()}
no_disco = {(c, f) for c in CLASSES for f in ALL[c]}
faltam, sobram = no_disco - set(GRUPO), set(GRUPO) - no_disco
print('Classes:', CLASSES, '| images on disk:', len(no_disco), '| images in the group file:', len(GRUPO))
assert not faltam and not sobram, f'The files on disk and in the CSV do not match ({len(faltam)} missing from the CSV, {len(sobram)} missing on disk). Example: {list(faltam)[:3]} {list(sobram)[:3]}'
print('Distinct groups:', len(set(GRUPO.values())))

def ic_t(x):
    x = np.asarray(x, float); n = len(x); m, sd = x.mean(), x.std(ddof=1)
    h = st.t.ppf(0.975, n - 1) * sd / np.sqrt(n); return m, sd, m - h, m + h

def wilson(k, n, z=1.959964):
    p = k / n; den = 1 + z * z / n; centro = (p + z * z / (2 * n)) / den
    meia = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return centro - meia, centro + meia

In [ ]:
# ======================= GROUPED PARTITION =======================
def split_agrupado(seed):
    """70/15/15 per cultivar, but drawing GROUPS of near-duplicate images (not single images)."""
    rng = random.Random(seed)
    out = {'train': [], 'val': [], 'test': []}
    for c in CLASSES:
        arquivos = ALL[c]; n = len(arquivos)
        n_tr, n_va = int(n * SPLIT_RATIO[0]), int(n * SPLIT_RATIO[1]); n_te = n - n_tr - n_va
        grupos = {}
        for f in arquivos: grupos.setdefault(GRUPO[(c, f)], []).append(f)
        ordem = sorted(grupos); rng.shuffle(ordem)
        destino = {'test': [], 'val': [], 'train': []}; livres = ordem
        for parte, alvo in (('test', n_te), ('val', n_va)):      # fill test and validation without exceeding the target
            cont, ainda = 0, []
            for g in livres:
                if cont + len(grupos[g]) <= alvo: destino[parte] += grupos[g]; cont += len(grupos[g])
                else: ainda.append(g)
            livres = ainda
        for g in livres: destino['train'] += grupos[g]
        for parte in out: out[parte] += [(c, f) for f in sorted(destino[parte])]
    return out

def split_antigo(seed):
    """Logic of the earlier 10 runs (image-by-image draw). Used only to compare the leakage."""
    random.seed(seed); out = {'train': [], 'val': [], 'test': []}
    for c in CLASSES:
        imgs = list(ALL[c]); random.shuffle(imgs)
        n_tr, n_va = int(len(imgs) * SPLIT_RATIO[0]), int(len(imgs) * SPLIT_RATIO[1])
        for parte, lst in (('train', imgs[:n_tr]), ('val', imgs[n_tr:n_tr + n_va]), ('test', imgs[n_tr + n_va:])):
            out[parte] += [(c, f) for f in lst]
    return out

def expostos(sp):
    """How many images of each subset have a copy (same group) in ANOTHER subset."""
    partes = {}
    for parte, lst in sp.items():
        for c, f in lst: partes.setdefault(GRUPO[(c, f)], set()).add(parte)
    return {p: sum(1 for c, f in sp[p] if len(partes[GRUPO[(c, f)]]) > 1) for p in sp}

SPLITS, linhas = {}, []
for seed in SEEDS:
    sp, antigo = split_agrupado(seed), split_antigo(seed)
    SPLITS[seed] = sp
    ex_novo, ex_antigo = expostos(sp), expostos(antigo)
    tam = tuple(len(sp[p]) for p in ('train', 'val', 'test'))
    assert sum(tam) == len(no_disco) and len(set(sp['train']) & set(sp['test'])) == 0
    assert ex_novo == {'train': 0, 'val': 0, 'test': 0}, f'Leakage in the grouped partition! seed={seed} {ex_novo}'
    pd.DataFrame([(c, f, GRUPO[(c, f)], p) for p in sp for c, f in sp[p]],
                 columns=['classe', 'arquivo', 'grupo', 'particao']).to_csv(os.path.join(BASE_WORK_DIR, f'split_agrupado_seed_{seed}.csv'), index=False)
    linhas.append(dict(seed=seed, train=tam[0], val=tam[1], test=tam[2],
                       teste_exposto_ANTIGO=ex_antigo['test'], teste_exposto_NOVO=ex_novo['test']))
T = pd.DataFrame(linhas); print(T.to_string(index=False))
print(f"\nExposed test images (earlier partition): mean {T.teste_exposto_ANTIGO.mean():.1f} of {T.test.mean():.0f} ({T.teste_exposto_ANTIGO.mean()/T.test.mean()*100:.1f}%)  |  grouped partition: {T.teste_exposto_NOVO.sum()}")
if not all((T.train == 1031) & (T.val == 220) & (T.test == 225)):
    print('WARNING: subset sizes are not exactly 1031/220/225 in every run; the paper must report the actual sizes.')
print('File lists of each run saved in:', BASE_WORK_DIR)

## Checkpoint
Check the output of the cell above (the table and the last two lines): every run must show zero exposed test images under the grouped partition and subset sizes of 1031 / 220 / 225. Training (Part B) starts below.

## Part B: training YOLOv8n-cls in the 10 runs (GPU)

In [ ]:
from ultralytics import YOLO
import ultralytics; print('Ultralytics', ultralytics.__version__)

resultados = []
for i, seed in enumerate(SEEDS):
    run_dir = os.path.join(BASE_WORK_DIR, f'run_seed_{seed}')
    resumo_json = os.path.join(run_dir, 'resumo.json')
    if os.path.exists(resumo_json):
        resultados.append(json.load(open(resumo_json))); print(f'--- Run {i+1}/10 (seed {seed}): already complete, skipping'); continue
    print(f'--- Run {i+1}/10 (seed {seed}) ---')
    sp = SPLITS[seed]; dataset_temp = os.path.join(run_dir, 'dataset_split')
    shutil.rmtree(dataset_temp, ignore_errors=True)
    for parte, lst in sp.items():
        for c, f in lst:
            d = os.path.join(dataset_temp, parte, c); os.makedirs(d, exist_ok=True)
            shutil.copy(os.path.join(DATASET_ORIGINAL, c, f), os.path.join(d, f))

    model = YOLO('yolov8n-cls.pt')
    model.train(data=dataset_temp, epochs=EPOCHS, patience=PATIENCE, imgsz=224, seed=seed,
                project=run_dir, name='treino', exist_ok=True, verbose=False)

    log = pd.read_csv(os.path.join(run_dir, 'treino', 'results.csv')); log.columns = [c.strip() for c in log.columns]
    acc = log['metrics/accuracy_top1'].values
    parada = len(log)
    melhor_1o_pico = int(acc.argmax()) + 1                      # first epoch with the maximum
    melhor_best_pt = int(np.where(acc == acc.max())[0].max()) + 1   # last tied epoch (the one saved as best.pt)

    m = YOLO(os.path.join(run_dir, 'treino', 'weights', 'best.pt')).val(
        data=dataset_temp, split='test', project=run_dir, name='aval_teste', exist_ok=True, verbose=False)
    top1 = float(m.top1); n_teste = len(sp['test']); acertos = round(top1 * n_teste)
    r = dict(run=i + 1, seed=seed, stopping_epoch=parada, best_epoch_first_max=melhor_1o_pico, best_pt_epoch=melhor_best_pt,
             n_epocas_empatadas=int((acc == acc.max()).sum()), n_test=n_teste, test_correct=acertos, test_top1=top1)
    json.dump(r, open(resumo_json, 'w')); resultados.append(r)
    print(f'Run {i+1} -> TEST accuracy: {top1:.4f} ({acertos}/{n_teste})')
    shutil.rmtree(dataset_temp, ignore_errors=True)       # the file lists are already saved in split_agrupado_seed_*.csv

In [ ]:
# ======================= RESULTS AND LATEX TABLE =======================
dfr = pd.DataFrame(resultados).sort_values('run')
dfr.to_csv(os.path.join(BASE_WORK_DIR, 'resumo_por_rodada.csv'), index=False)

m, sd, lo, hi = ic_t(dfr.test_top1)
ref = dfr[dfr.seed == REF_SEED].iloc[0]; wl, wh = wilson(int(ref.test_correct), int(ref.n_test))
print(dfr.to_string(index=False))
print(f'\nYOLOv8n-cls, GROUPED partition: {m:.4f} ± {sd:.4f}  95% t-interval [{lo:.4f}, {hi:.4f}]')
print(f'Reference run (seed {REF_SEED}): {int(ref.test_correct)}/{int(ref.n_test)} = {ref.test_top1:.4f}  95% Wilson interval [{wl:.4f}, {wh:.4f}]')
print('Earlier image-level partition (for comparison): 0.9911 ± 0.0063  95% t-interval [0.9866, 0.9956]  | reference 224/225 = 0.9956')

linhas = ''.join(f"{int(r.run)}{' (Reference)' if r.seed == REF_SEED else ''} & {int(r.seed)} & {int(r.stopping_epoch)} / 30 & {int(r.best_epoch_first_max)} & {int(r.best_pt_epoch)} & {r.test_top1:.4f} \\\\\n" for r in dfr.itertuples())
print('\n--- ROWS OF TABLE 4 (LaTeX) ---'); print(linhas)
print(f'Mean ± SD: stopping epoch {dfr.stopping_epoch.mean():.1f} ± {dfr.stopping_epoch.std():.1f} | first maximum {dfr.best_epoch_first_max.mean():.1f} ± {dfr.best_epoch_first_max.std():.1f} | best.pt {dfr.best_pt_epoch.mean():.1f} ± {dfr.best_pt_epoch.std():.1f} | accuracy {m:.4f} ± {sd:.4f}')

## Part C: color baselines (RBF-SVM and Random Forest) on the SAME grouped partitions
Features: HSV histograms (36 + 32 + 32 bins) plus the mean and standard deviation of each Lab channel (106 features), computed from the pixels of the image whose maximum channel value exceeds 5 (a brightness threshold that excludes only near-black pixels; it is not a stalk segmentation, and it retains most of each cropped image, including the dark base). `StandardScaler` fitted on the training subset only, SVM `C` chosen on the validation subset from {1, 10, 100}, `gamma='scale'`; Random Forest with 300 trees. Two views are used: the whole cropped image (longer side limited to 1,200 pixels) and the central 224 x 224 view that the network receives at test time. No GPU is needed.

In [ ]:
import cv2
from PIL import Image
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
CROP = 224

def color_features(rgb, mask):
    if mask.sum() < 50: return np.zeros(36 + 32 + 32 + 6)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[mask]; lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)[mask].astype(float)
    hs = []
    for ch, bins, rng in ((0, 36, (0, 180)), (1, 32, (0, 256)), (2, 32, (0, 256))):
        h = np.histogram(hsv[:, ch], bins=bins, range=rng)[0].astype(float); hs.append(h / max(h.sum(), 1))
    return np.concatenate(hs + [lab.mean(0), lab.std(0)])

def load_rgb_mask(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im.ndim == 3 and im.shape[2] == 4: mask = im[:, :, 3] > 0; bgr = im[:, :, :3]
    else:
        bgr = im if im.ndim == 3 else cv2.cvtColor(im, cv2.COLOR_GRAY2BGR); mask = bgr.max(2) > 5
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), mask

def center_view(rgb, mask, size=CROP):
    h, w = rgb.shape[:2]; s = size / min(h, w); nh, nw = max(size, round(h * s)), max(size, round(w * s))
    im = np.array(Image.fromarray(rgb).resize((nw, nh), Image.BILINEAR))
    mk = np.array(Image.fromarray(mask.astype(np.uint8) * 255).resize((nw, nh), Image.NEAREST)) > 0
    t, l = (nh - size) // 2, (nw - size) // 2
    return im[t:t + size, l:l + size], mk[t:t + size, l:l + size]

def extract(c, f):
    rgb, mask = load_rgb_mask(os.path.join(DATASET_ORIGINAL, c, f))
    h, w = rgb.shape[:2]; k = min(1.0, 1200 / max(h, w))
    if k < 1:
        rgb_s = cv2.resize(rgb, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
        mask_s = cv2.resize(mask.astype(np.uint8), None, fx=k, fy=k, interpolation=cv2.INTER_NEAREST) > 0
    else: rgb_s, mask_s = rgb, mask
    cim, cmk = center_view(rgb, mask)
    return color_features(rgb_s, mask_s), color_features(cim, cmk), float(mask.mean())

keys = [(c, f) for c in CLASSES for f in ALL[c]]
cache_path = os.path.join(BASE_WORK_DIR, 'features_cor.npz'); z = None
if os.path.exists(cache_path):
    z = np.load(cache_path, allow_pickle=True)
    if list(map(tuple, z['keys'])) != keys: z = None
if z is None:
    FULL, CEN, MF = [], [], []
    for n, (c, f) in enumerate(keys):
        a, b, mf = extract(c, f); FULL.append(a); CEN.append(b); MF.append(mf)
        if n % 200 == 0: print(f'  {n}/{len(keys)}')
    FULL, CEN, MF = np.array(FULL), np.array(CEN), np.array(MF)
    np.savez(cache_path, keys=np.array(keys, dtype=object), full=FULL, cen=CEN, maskfrac=MF)
else: FULL, CEN, MF = z['full'], z['cen'], z['maskfrac']
IDX = {k: i for i, k in enumerate(keys)}
print('Features:', FULL.shape, CEN.shape)
# Diagnostic: what fraction of the pixels does the mask keep? Close to 1.0 means that the background is NOT being excluded.
print('Mean fraction of pixels inside the mask, per cultivar:')
print(pd.Series(MF, index=[c for c, f in keys]).groupby(level=0).mean().round(3).to_string())

In [ ]:
def run_models(Xtr, ytr, Xva, yva, Xte, yte, seed):
    sc = StandardScaler().fit(Xtr); Xtr, Xva, Xte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)
    best_c, best_va = None, -1
    for C in (1, 10, 100):
        a = SVC(C=C, kernel='rbf', gamma='scale').fit(Xtr, ytr).score(Xva, yva)
        if a > best_va: best_c, best_va = C, a
    svm = SVC(C=best_c, kernel='rbf', gamma='scale').fit(Xtr, ytr)
    rf = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1).fit(Xtr, ytr)
    out = {}
    for name, mdl in (('SVM', svm), ('RF', rf)):
        p = mdl.predict(Xte)
        out[name] = dict(acc=float((p == yte).mean()), correct=int((p == yte).sum()), extra=best_c if name == 'SVM' else None)
    return out

CI = {c: i for i, c in enumerate(CLASSES)}
rows = []
for i, seed in enumerate(SEEDS):
    sp = SPLITS[seed]
    ix = {s: np.array([IDX[k] for k in v]) for s, v in sp.items()}
    y = {s: np.array([CI[c] for c, f in v]) for s, v in sp.items()}
    for view, M in (('tolete_inteiro', FULL), ('recorte_central_224', CEN)):
        res = run_models(M[ix['train']], y['train'], M[ix['val']], y['val'], M[ix['test']], y['test'], seed)
        for model, r in res.items():
            rows.append(dict(run=i + 1, seed=seed, view=view, model=model, test_top1=r['acc'], test_correct=r['correct'], svm_C=r['extra']))
    print(f'Run {i+1}/10 (seed {seed}):', {f"{r['view'][:5]}-{r['model']}": round(r['test_top1'], 4) for r in rows[-4:]})

dfb = pd.DataFrame(rows); dfb.to_csv(os.path.join(BASE_WORK_DIR, 'baseline_cor_por_rodada.csv'), index=False)
print('\nCOLOR BASELINE, GROUPED partition, 10 runs')
for (view, model), g in dfb.groupby(['view', 'model']):
    m, sd, lo, hi = ic_t(g.test_top1); print(f'{view:22s} {model:4s} {m:.4f} ± {sd:.4f}  95% interval [{lo:.4f}, {hi:.4f}]')
print('Earlier image-level partition (for comparison), color SVM: 0.9609 ± 0.0173')

Output files, written to the `runs_agrupado` folder: `split_agrupado_seed_*.csv` (subset of every image in each run), `run_seed_*/resumo.json` and `resumo_por_rodada.csv` (stopping epoch, best epochs, and test accuracy of each run), `features_cor.npz` (color features), and `baseline_cor_por_rodada.csv` (color baselines). The post-hoc analyses of the trained models are in `Partes_D_E_figuras_e_metricas.ipynb`.